# Netflix Prize — Item-Item kNN (model 2)

Second model for the mandatory comparison: **item-based collaborative filtering**.
Reuses the exact parse / per-user temporal split / baseline / metrics from the MF
notebook, so MF vs kNN is apples-to-apples (same `SEED`, same train/test).

Pipeline: baseline residuals → item-item cosine similarity (shrinkage, power-user
cap, blockwise top-K) → neighborhood prediction → RMSE + MAP@10 → compare.

**Colab:** GPU not needed (this is numpy/scipy). High-RAM helps for the full set.

## 0. Setup & config

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# Same DATA_DIR / cache as the MF notebook (so the parse is instant if already cached).
DATA_DIR   = '/content/netflix'      # <-- EDIT ME
CACHE_NPZ  = os.path.join(DATA_DIR, 'ratings_cache.npz')

SUBSAMPLE_FILES = [1, 2, 3, 4]
CHUNK_ROWS      = 5_000_000
USER_FRAC = 1.0          # 0.35 fits standard RAM; kNN similarity is the heavy step
SEED      = 42
TEST_FRAC = 0.20
RELEVANT_THRESHOLD = 3.5

# kNN hyperparameters
K_NEIGHBORS = 50         # neighbors kept per item
SHRINK      = 100.0      # similarity shrinkage: s *= n_ij / (n_ij + SHRINK)
DENOM_REG   = 1.0        # prediction denom shrinkage: corr = num / (Sigma|s| + DENOM_REG)
RCAP        = 300        # cap ratings/user for the SIMILARITY step (bounds cost)
SIM_BLOCK   = 512        # items per block when computing similarities

rng = np.random.default_rng(SEED)
print('config loaded')

In [ ]:
# --- Colab only: mount Drive / unzip. Skip locally. ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # !mkdir -p /content/netflix
    # !unzip -o '/content/drive/MyDrive/archive.zip' -d /content/netflix
except Exception as e:
    print('Not on Colab (or skip):', e)

## 1. Parse + map ids + split (shared with the MF notebook)

In [ ]:
def parse_combined(path, chunksize=CHUNK_ROWS):
    us, ms, rs, ds = [], [], [], []
    cur = 0
    for chunk in pd.read_csv(path, header=None, names=['user', 'rating', 'date'],
                             dtype={'user': str}, chunksize=chunksize):
        is_hdr = chunk['rating'].isna()
        mv = pd.to_numeric(chunk['user'].where(is_hdr).str.rstrip(':'),
                           errors='coerce').ffill().fillna(cur)
        rows = ~is_hdr
        us.append(pd.to_numeric(chunk.loc[rows, 'user']).to_numpy(np.int32))
        ms.append(mv[rows].to_numpy(np.int16))
        rs.append(chunk.loc[rows, 'rating'].to_numpy(np.int8))
        ds.append(chunk.loc[rows, 'date'].to_numpy('datetime64[D]'))
        cur = int(mv.iloc[-1])
    return (np.concatenate(us), np.concatenate(ms),
            np.concatenate(rs), np.concatenate(ds))

if os.path.exists(CACHE_NPZ):
    z = np.load(CACHE_NPZ)
    raw_user, movie, rating, date = z['user'], z['movie'], z['rating'], z['date']
    print(f'loaded cache: {len(rating):,} ratings')
else:
    pu, pm, pr, pd_ = [], [], [], []
    for n in SUBSAMPLE_FILES:
        u, m, r, d = parse_combined(os.path.join(DATA_DIR, f'combined_data_{n}.txt'))
        pu.append(u); pm.append(m); pr.append(r); pd_.append(d)
    raw_user = np.concatenate(pu); movie = np.concatenate(pm)
    rating = np.concatenate(pr); date = np.concatenate(pd_)
    np.savez(CACHE_NPZ, user=raw_user, movie=movie, rating=rating, date=date)
    print(f'parsed & cached {len(rating):,} ratings')

In [ ]:
uniq_users, user = np.unique(raw_user, return_inverse=True)
user = user.astype(np.int32); del raw_user
movie = (movie - 1).astype(np.int16)
if USER_FRAC < 1.0:
    keep = np.zeros(len(uniq_users), bool)
    keep[rng.choice(len(uniq_users), int(len(uniq_users) * USER_FRAC), replace=False)] = True
    mrows = keep[user]
    user, movie, rating, date = user[mrows], movie[mrows], rating[mrows], date[mrows]
    _, user = np.unique(user, return_inverse=True); user = user.astype(np.int32)
n_users  = int(user.max()) + 1
n_movies = int(movie.max()) + 1
print(f'{len(rating):,} ratings | {n_users:,} users | {n_movies:,} movies')

titles = pd.read_csv(os.path.join(DATA_DIR, 'movie_titles.csv'),
                     header=None, encoding='latin-1', on_bad_lines='skip',
                     names=['movie_id', 'year', 'title'], usecols=[0, 1, 2])
title_of = dict(zip(titles['movie_id'].astype(int), titles['title']))

In [ ]:
order = np.lexsort((date, user))
su = user[order]
_, starts, counts = np.unique(su, return_index=True, return_counts=True)
starts = starts.astype(np.int32); counts = counts.astype(np.int32)
pos    = np.arange(len(order), dtype=np.int32) - np.repeat(starts, counts)
n_test = np.where(counts >= 5, np.ceil(counts * TEST_FRAC).astype(np.int32), 0)
thresh = np.repeat(counts - n_test, counts)
is_test = np.zeros(len(order), bool)
is_test[order] = pos >= thresh
del order, su, pos, thresh, starts, counts
tr = ~is_test
u_tr, m_tr, r_tr = user[tr], movie[tr], rating[tr]
u_te, m_te, r_te = user[is_test], movie[is_test], rating[is_test]
print(f'train {tr.sum():,}  |  test {is_test.sum():,}')

## 2. Baseline biases (centering + comparison point)

In [ ]:
def fit_biases(u, m, r, n_u, n_m, reg=10.0, n_iter=10):
    mu = r.mean()
    bu = np.zeros(n_u, np.float32); bi = np.zeros(n_m, np.float32)
    uc = np.bincount(u, minlength=n_u); ic = np.bincount(m, minlength=n_m)
    for _ in range(n_iter):
        bi = np.bincount(m, weights=r - mu - bu[u], minlength=n_m) / (ic + reg)
        bu = np.bincount(u, weights=r - mu - bi[m], minlength=n_u) / (uc + reg)
    return mu, bu, bi

mu, bu, bi = fit_biases(u_tr, m_tr, r_tr, n_users, n_movies)
def predict_baseline(u, m):
    return np.clip(mu + bu[u] + bi[m], 1, 5)
def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))
print('baseline test RMSE:', rmse(predict_baseline(u_te, m_te), r_te))

## 3. Item-item similarity

Cosine similarity on **baseline residuals** `r - (μ + b_u + b_i)` (removes user/item
offsets so similarity reflects *taste*, not popularity). Power users are capped at
`RCAP` ratings for this step — a few users rate thousands of movies and dominate the
`Σ_u d_u²` cost. Similarity is shrunk by co-rating count `n_ij/(n_ij+SHRINK)` so pairs
with few common users aren't trusted. Top-`K` neighbors per item are kept; the full
17.7k×17.7k matrix is never materialized (computed in blocks).

In [ ]:
resid_tr = (r_tr - predict_baseline(u_tr, m_tr)).astype(np.float32)

# user profiles sorted by user (used for BOTH capping and prediction)
ords = np.argsort(u_tr, kind='stable')
su = u_tr[ords]
ub = np.searchsorted(su, np.arange(n_users + 1))      # CSR-style row bounds
prof_items = m_tr[ords].astype(np.int32)
prof_resid = resid_tr[ords]

# cap power users (drop random surplus rows beyond RCAP) for the similarity matrix
counts = ub[1:] - ub[:-1]
mask = np.ones(len(u_tr), bool)
for uidx in np.where(counts > RCAP)[0]:
    s, e = ub[uidx], ub[uidx + 1]
    drop = s + rng.choice(e - s, (e - s) - RCAP, replace=False)
    mask[drop] = False
us_c, ms_c, rs_c = su[mask], prof_items[mask], prof_resid[mask]
print(f'similarity uses {len(us_c):,} of {len(u_tr):,} ratings (cap {RCAP}/user)')

R  = csr_matrix((rs_c, (us_c, ms_c)), shape=(n_users, n_movies))
Bn = csr_matrix((np.ones(len(rs_c), np.float32), (us_c, ms_c)), shape=(n_users, n_movies))
col_norm = np.sqrt(np.asarray(R.multiply(R).sum(0)).ravel()) + 1e-8
Rt, Bt = R.T.tocsr(), Bn.T.tocsr()

In [ ]:
nbr    = np.zeros((n_movies, K_NEIGHBORS), np.int32)
nbrsim = np.zeros((n_movies, K_NEIGHBORS), np.float32)
t0 = time.time()
for start in range(0, n_movies, SIM_BLOCK):
    end = min(start + SIM_BLOCK, n_movies)
    num = (Rt[start:end] @ R).toarray()              # resid . resid
    cc  = (Bt[start:end] @ Bn).toarray()             # co-rating counts
    denom = col_norm[start:end, None] * col_norm[None, :]
    sim = (num / denom) * (cc / (cc + SHRINK))       # cosine x shrinkage
    sim[np.arange(end - start), np.arange(start, end)] = -np.inf   # drop self
    part = np.argpartition(-sim, K_NEIGHBORS, axis=1)[:, :K_NEIGHBORS]
    psim = np.take_along_axis(sim, part, axis=1)
    o = np.argsort(-psim, axis=1)
    nbr[start:end]    = np.take_along_axis(part, o, axis=1)
    nbrsim[start:end] = np.take_along_axis(psim, o, axis=1)
nbrsim[~np.isfinite(nbrsim)] = 0.0
nbrsim[nbrsim < 0] = 0.0                              # keep positive correlations
print(f'similarity done in {time.time()-t0:.0f}s | mean |neighbors|>0 per item: '
      f'{(nbrsim > 0).sum(1).mean():.1f}')

## 4. Neighborhood prediction

`r̂_ui = baseline_ui + (Σ_{j∈N(i), rated by u} s_ij·resid_uj) / (Σ s_ij + DENOM_REG)`.

The `DENOM_REG` term is **denominator shrinkage**: without it, a candidate whose only
rated neighbors carry tiny similarity mass gets a wild weighted-average residual and
the correction *hurts* (raw kNN scored worse than the baseline). Adding `DENOM_REG`
pulls low-confidence corrections toward 0, so kNN beats the baseline (tuned: ~0.925→
0.915 on a subsample). Vectorized per user with a reusable buffer — cost is
`Σ_u (d_u + |candidates_u|·K)`, not `n_users·n_movies`.

In [ ]:
_buf_resid = np.zeros(n_movies, np.float32)
_buf_rated = np.zeros(n_movies, bool)

def predict_knn(u_arr, m_arr):
    u_arr = np.asarray(u_arr); m_arr = np.asarray(m_arr)
    base = predict_baseline(u_arr, m_arr)
    out = base.astype(np.float32).copy()
    o = np.argsort(u_arr, kind='stable')
    uu = u_arr[o]
    gusers, gstart = np.unique(uu, return_index=True)
    gend = np.append(gstart[1:], len(uu))
    for g, uidx in enumerate(gusers):
        rows = o[gstart[g]:gend[g]]            # positions in the original arrays
        items = prof_items[ub[uidx]:ub[uidx + 1]]
        resid = prof_resid[ub[uidx]:ub[uidx + 1]]
        _buf_resid[items] = resid; _buf_rated[items] = True
        C = m_arr[rows]
        nb = nbr[C]; ns = nbrsim[C]            # (|C| x K)
        rated = _buf_rated[nb]
        numv = (ns * _buf_resid[nb] * rated).sum(1)
        denv = (ns * rated).sum(1)             # ns already >= 0
        corr = numv / (denv + DENOM_REG)       # denom shrinkage damps small-mass corrections
        out[rows] = np.clip(base[rows] + corr, 1, 5)
        _buf_resid[items] = 0; _buf_rated[items] = False
    return out

## 5. Evaluation — RMSE + MAP@10 (same metrics as MF notebook)

In [ ]:
def map_at_k(u_te, m_te, r_te, predict_fn, k=10, max_users=20000, seed=SEED):
    pred = predict_fn(u_te, m_te)
    o = np.argsort(u_te, kind='stable')
    uu, rr, pp = u_te[o], r_te[o], pred[o]
    users_u, st = np.unique(uu, return_index=True); en = np.append(st[1:], len(uu))
    rgn = np.random.default_rng(seed); sel = np.arange(len(users_u))
    if len(sel) > max_users: sel = rgn.choice(sel, max_users, replace=False)
    aps = []
    for j in sel:
        s, e = st[j], en[j]
        if e - s < 2: continue
        rel = rr[s:e] >= RELEVANT_THRESHOLD; R = int(rel.sum())
        if R == 0: continue
        rank = np.argsort(-pp[s:e], kind='stable')[:k]
        hits = 0; ap = 0.0
        for i, ri in enumerate(rank, 1):
            if rel[ri]: hits += 1; ap += hits / i
        aps.append(ap / min(k, R))
    return float(np.mean(aps)), len(aps)

def map_at_k_sampled(predict_fn, u_tr, m_tr, u_te, m_te, r_te,
                     k=10, n_neg=100, max_users=10000, seed=SEED):
    rgn = np.random.default_rng(seed)
    o = np.argsort(u_tr, kind='stable'); su, sm = u_tr[o], m_tr[o]
    bounds = np.searchsorted(su, np.arange(n_users + 1))
    ot = np.argsort(u_te, kind='stable'); tu, tm, trr = u_te[ot], m_te[ot], r_te[ot]
    tusers, tst = np.unique(tu, return_index=True); ten = np.append(tst[1:], len(tu))
    sel = np.arange(len(tusers))
    if len(sel) > max_users: sel = rgn.choice(sel, max_users, replace=False)
    aps = []
    for j in sel:
        uidx = int(tusers[j]); s, e = tst[j], ten[j]
        t_movies, t_r = tm[s:e], trr[s:e]
        rel = t_r >= RELEVANT_THRESHOLD; R = int(rel.sum())
        if R == 0: continue
        seen = set(sm[bounds[uidx]:bounds[uidx + 1]].tolist()) | set(t_movies.tolist())
        negs = []
        while len(negs) < n_neg:
            for c in rgn.integers(0, n_movies, n_neg):
                c = int(c)
                if c not in seen:
                    negs.append(c); seen.add(c)
                    if len(negs) >= n_neg: break
        cand = np.concatenate([t_movies, np.array(negs, dtype=np.int32)])
        labels = np.concatenate([rel, np.zeros(len(negs), bool)])
        scores = predict_fn(np.full(len(cand), uidx, np.int32), cand)
        rank = np.argsort(-scores, kind='stable')[:k]
        hits = 0; ap = 0.0
        for i, ri in enumerate(rank, 1):
            if labels[ri]: hits += 1; ap += hits / i
        aps.append(ap / min(k, R))
    return float(np.mean(aps)), len(aps)

In [ ]:
t0 = time.time()
pred_knn_te = predict_knn(u_te, m_te)      # full test (the heavy call, ~1-2 min)
print(f'kNN test RMSE: {rmse(pred_knn_te, r_te):.4f}  ({time.time()-t0:.0f}s)')

# reuse precomputed preds for held-out MAP (avoid recomputing); sampled-neg needs live calls
s_ho, n_ho = map_at_k(u_te, m_te, r_te, lambda u, m: pred_knn_te)
s_sn, n_sn = map_at_k_sampled(predict_knn, u_tr, m_tr, u_te, m_te, r_te)
print(f'kNN MAP@10  held-out={s_ho:.4f} ({n_ho})  sampled-neg={s_sn:.4f} ({n_sn})')
s_b_ho, _ = map_at_k(u_te, m_te, r_te, predict_baseline)
s_b_sn, _ = map_at_k_sampled(predict_baseline, u_tr, m_tr, u_te, m_te, r_te)
print(f'baseline MAP@10  held-out={s_b_ho:.4f}  sampled-neg={s_b_sn:.4f}')

## 6. Comparison & notes

Fill the MF column from the MF notebook for the report's comparison table:

| Model | RMSE | MAP@10 (sampled-neg) | Training cost | Notes |
|---|---|---|---|---|
| Baseline (bias) | ~0.93 | ~0.18 | trivial | popularity floor |
| Item-kNN | *(this nb)* | *(this nb)* | `Σ_u d_u²` similarity, capped | localized taste, interpretable |
| MF (latent) | 0.866 | 0.311 | SGD, GPU | best single model |

**Talking points for the report (training complexity / usability — rubric item C):**
- kNN similarity is `O(Σ_u d_u²)` — dominated by power users, hence the `RCAP` cap.
- kNN is **interpretable**: "recommended because you liked *neighbor X*" (ties into
  the optional Explainable-Recommendations task).
- kNN and MF make *decorrelated* errors (local vs. global structure) — adding the kNN
  prediction as a column in the MF notebook's ridge blend should beat MF alone.
- Try `SHRINK`, `RCAP`, `K_NEIGHBORS` sensitivity for the report.